In [39]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim

In [40]:
# Constants
RandomSTATE = 1082
TEST_SIZE = 0.2

In [41]:
# Load real data
df = pd.read_csv("../../Datasets/csv/spoofing-merged-gps-only.csv")
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols).values
labels = df['label']

In [42]:
# Data Preprocessing
if labels.dtype == object or labels.dtype.name == 'category':
    le = LabelEncoder()
    y = le.fit_transform(labels)
else:
    y = labels.values

n_classes = len(np.unique(y))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=1082, stratify=y)

### Two Neural Networks Compete

- Generator: This network tries to create fake data that looks like your real data.
- Discriminator: This network tries to tell the difference between real data (from your dataset) and fake data (from the generator).

### The Training Loop

- In each round (epoch), the generator produces some fake data.
- The discriminator looks at a mix of real and fake data and tries to classify each as real or fake.
- Both networks get feedback: the generator learns how to make better fakes, and the discriminator learns how to get better at spotting them.
- Both networks update their parameters using this feedback.

### The Loss Values

- Discriminator Loss (Loss_D): How well the discriminator is doing. Lower means is better.
- Generator Loss (Loss_G): How well the generator is fooling the discriminator. Lower means it's making more convincing fakes.
- Ideally, neither network gets too good or too bad. If one "wins" completely, the other can't learn.

In [43]:
# Define GAN
class Generator(nn.Module):
    def __init__(self, noise_dim, label_dim, data_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim + label_dim, 128),
            nn.ReLU(),
            nn.Linear(128, data_dim)
        )

    def forward(self, z, labels_onehot):
        x = torch.cat([z, labels_onehot], dim=1)
        return self.model(x)


class Discriminator(nn.Module):
    def __init__(self, label_dim, data_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(data_dim + label_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x, labels_onehot):
        x = torch.cat([x, labels_onehot], dim=1)
        return self.model(x)


In [44]:
# Hyperparameters
noise_dim = 100
label_dim = n_classes
data_dim = X_train.shape[1]
epochs = 1000
batch_size = 128
lr = 0.0002
betas = (0.5, 0.999)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
G = Generator(noise_dim, label_dim, data_dim).to(device)
D = Discriminator(label_dim, data_dim).to(device)
criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=betas)
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=betas)

In [45]:
# DataLoader
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [46]:
# Training Loop
for epoch in range(epochs):
    for real_data, real_labels in train_loader:
        real_data = real_data.to(device)
        real_labels = real_labels.to(device)
        batch_sz = real_data.size(0)
        onehot_real = torch.nn.functional.one_hot(real_labels, num_classes=label_dim).float()

        # Train Discriminator
        optimizer_D.zero_grad()
        labels_real = torch.ones((batch_sz, 1), device=device)
        output_real = D(real_data, onehot_real)
        loss_real = criterion(output_real, labels_real)

        # Fake data
        noise = torch.randn(batch_sz, noise_dim, device=device)
        fake_labels = torch.randint(0, label_dim, (batch_sz,), device=device)
        onehot_fake = torch.nn.functional.one_hot(fake_labels, num_classes=label_dim).float()
        fake_data = G(noise, onehot_fake)
        labels_fake = torch.zeros((batch_sz, 1), device=device)
        output_fake = D(fake_data.detach(), onehot_fake)
        loss_fake = criterion(output_fake, labels_fake)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        # Train Generator
        optimizer_G.zero_grad()
        output_fake = D(fake_data, onehot_fake)
        loss_G = criterion(output_fake, labels_real)
        loss_G.backward()
        optimizer_G.step()

    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{epochs} | Loss_D: {loss_D.item():.4f} | Loss_G: {loss_G.item():.4f}")

Epoch 1/1000 | Loss_D: 1.3502 | Loss_G: 0.6485
Epoch 20/1000 | Loss_D: 1.0032 | Loss_G: 1.0026
Epoch 40/1000 | Loss_D: 0.9773 | Loss_G: 1.1746
Epoch 60/1000 | Loss_D: 0.7960 | Loss_G: 1.5322
Epoch 80/1000 | Loss_D: 0.6728 | Loss_G: 1.6675
Epoch 100/1000 | Loss_D: 0.8193 | Loss_G: 1.4974
Epoch 120/1000 | Loss_D: 0.8324 | Loss_G: 1.5218
Epoch 140/1000 | Loss_D: 0.6955 | Loss_G: 1.8086
Epoch 160/1000 | Loss_D: 0.7783 | Loss_G: 1.6034
Epoch 180/1000 | Loss_D: 0.6780 | Loss_G: 1.8728
Epoch 200/1000 | Loss_D: 0.8501 | Loss_G: 1.6942
Epoch 220/1000 | Loss_D: 0.7860 | Loss_G: 1.8353
Epoch 240/1000 | Loss_D: 0.8320 | Loss_G: 1.5744
Epoch 260/1000 | Loss_D: 0.7397 | Loss_G: 1.7130
Epoch 280/1000 | Loss_D: 0.8870 | Loss_G: 1.4236
Epoch 300/1000 | Loss_D: 0.9142 | Loss_G: 1.5450
Epoch 320/1000 | Loss_D: 0.7687 | Loss_G: 1.8535
Epoch 340/1000 | Loss_D: 0.8030 | Loss_G: 1.6685
Epoch 360/1000 | Loss_D: 0.7723 | Loss_G: 1.7339
Epoch 380/1000 | Loss_D: 0.8964 | Loss_G: 1.7451
Epoch 400/1000 | Loss_D: 0

In [47]:
# Generate synthetic data
G.eval()
num_synth = 10000
with torch.no_grad():
    synth_labels = np.random.choice(np.arange(label_dim), num_synth)
    synth_labels_tensor = torch.tensor(synth_labels, dtype=torch.long, device=device)
    synth_labels_onehot = torch.nn.functional.one_hot(synth_labels_tensor, num_classes=label_dim).float()
    z = torch.randn(num_synth, noise_dim, device=device)
    generated = G(z, synth_labels_onehot).cpu().numpy()
    generated = scaler.inverse_transform(generated)

# Save synthetic data to CSV
synthetic_df = pd.DataFrame(generated, columns=df.drop(columns=drop_cols).columns)
if 'le' in locals():
    synthetic_df['label'] = le.inverse_transform(synth_labels)
else:
    synthetic_df['label'] = synth_labels
synthetic_df.to_csv("syntheticSpoofing_gan_data.csv", index=False)
print("Synthetic data saved to syntheticSpoofing_gan_data.csv")

Synthetic data saved to syntheticSpoofing_gan_data.csv
